# 2. Toffoli arithmetic

Why does a fault-tolerant cost count only one kind of gate, and what does it cost to add two numbers, compare them, or rotate a qubit by an angle held in a register?

In [1]:
import numpy as np
import pennylane.estimator as qre

from qcost.arithmetic import adder, comparator, phase_gradient_rotation, rotation_synthesis_t
from qcost.circuit import all_inputs, read_int


def pennylane_toffolis(op):
    counts = {str(k): v for k, v in qre.estimate(op).gate_counts.items()}
    return counts.get("Toffoli", 0)

## The currency

In a surface code, Clifford gates (H, S, CNOT, Pauli measurements) are done by moving and measuring patches of qubits; they cost time and space but no special resource. A non-Clifford gate needs a magic state, made in a factory that takes thousands of physical qubits and tens of rounds per state (notebook 06). So the cost of an algorithm is, to a first approximation, its count of non-Clifford gates: T gates, or Toffolis, made from a CCZ state each.

The basic unit of this repository is Gidney's temporary AND [arXiv:1709.06648](https://arxiv.org/abs/1709.06648), Fig. 3: a Toffoli onto a qubit known to be $|0\rangle$, costing 4 T gates. Its undo is free: measure the target in the X basis, and if the outcome is 1, fix the sign with a CZ between the two controls. Here is that fix on a three-qubit state vector, controls $a, b$ and target $t = a \wedge b$:

In [2]:
rng = np.random.default_rng(0)
amp = rng.normal(size=4) + 1j * rng.normal(size=4)
amp /= np.linalg.norm(amp)
start = np.zeros(8, complex)            # qubit order a, b, t
for ab in range(4):
    a, b = ab >> 1, ab & 1
    start[(a << 2) | (b << 1) | (a & b)] = amp[ab]

plus, minus = np.array([1, 1]) / np.sqrt(2), np.array([1, -1]) / np.sqrt(2)
for outcome, v in [(0, plus), (1, minus)]:
    after = np.einsum("abt,t->ab", start.reshape(2, 2, 2), v.conj()).ravel()
    after /= np.linalg.norm(after)
    if outcome == 1:
        after = after * np.array([1, 1, 1, -1])  # CZ on a, b
    print(f"outcome {outcome}: controls left in their original state:", np.allclose(after, amp))

outcome 0: controls left in their original state: True
outcome 1: controls left in their original state: True


Both outcomes leave the controls exactly as they were, with only Clifford gates. Compute costs one Toffoli, uncompute costs none.

The package's `Circuit` has five gates: `x`, `cx`, `and_` (one Toffoli), `unand` (free), `cswap` (one Toffoli). All of them send basis states to basis states (up to the sign fix), so a circuit can be checked on every input at once by pushing bit strings through it, and its cost is just the number of `and_`, `ccx` and `cswap` gates.

## Addition: $n - 1$ Toffolis

$b \leftarrow a + b \bmod 2^n$. The carry into bit $i+1$ is the majority of $a_i$, $b_i$ and the carry $c_i$, and the majority is one AND after two CNOTs: $c_{i+1} = (a_i\oplus c_i)(b_i\oplus c_i)\oplus c_i$. Ripple up computing carries, then ripple down undoing them for free while writing the sum bits:

In [3]:
c, a, b = adder(3)
for g in c.gates:
    print(g)
print("Toffolis:", c.toffolis())

('and', 0, 3, 6)
('cx', 6, 1)
('cx', 6, 4)
('and', 1, 4, 7)
('cx', 6, 7)
('cx', 7, 5)
('cx', 2, 5)
('cx', 6, 7)
('unand', 1, 4, 7)
('cx', 6, 1)
('cx', 1, 4)
('unand', 0, 3, 6)
('cx', 0, 3)
Toffolis: 2


In [4]:
for n in range(1, 7):
    c, a, b = adder(n)
    X = all_inputs([a, b], c.n)          # every (a, b): 4^n rows
    Y = c.run(X)
    ok = (read_int(Y, b) == (read_int(X, a) + read_int(X, b)) % 2**n).all()
    print(f"n = {n}: {len(X):5d} inputs correct: {ok}; Toffolis {c.toffolis()},"
          f" PennyLane SemiAdder {pennylane_toffolis(qre.SemiAdder(n))}")

n = 1:     4 inputs correct: True; Toffolis 0, PennyLane SemiAdder 0
n = 2:    16 inputs correct: True; Toffolis 1, PennyLane SemiAdder 1
n = 3:    64 inputs correct: True; Toffolis 2, PennyLane SemiAdder 2
n = 4:   256 inputs correct: True; Toffolis 3, PennyLane SemiAdder 3
n = 5:  1024 inputs correct: True; Toffolis 4, PennyLane SemiAdder 4
n = 6:  4096 inputs correct: True; Toffolis 5, PennyLane SemiAdder 5


$n - 1$ Toffolis, i.e. $4n - 4$ T gates [Gidney, Fig. 1], and PennyLane's resource estimator counts the same.

## Comparison: $n$ Toffolis

To test $a < b$, note that $(2^n - 1 - a) + b \ge 2^n$ exactly when $b > a$: the carry out of the sum of $\bar a$ (all bits of $a$ flipped) and $b$ is the answer. So compute only the carries, $n$ ANDs, copy the last one into a flag, and undo the carries by measurement:

In [5]:
for n in range(1, 7):
    c, a, b, flag = comparator(n)
    X = all_inputs([a, b], c.n)
    Y = c.run(X)
    ok = (Y[:, flag] == (read_int(X, a) < read_int(X, b))).all()
    print(f"n = {n}: correct on all {len(X)} inputs: {ok}; Toffolis {c.toffolis()},"
          f" PennyLane RegisterComparator {pennylane_toffolis(qre.RegisterComparator(n, n))}")

n = 1: correct on all 4 inputs: True; Toffolis 1, PennyLane RegisterComparator 1
n = 2: correct on all 16 inputs: True; Toffolis 2, PennyLane RegisterComparator 3
n = 3: correct on all 64 inputs: True; Toffolis 3, PennyLane RegisterComparator 5
n = 4: correct on all 256 inputs: True; Toffolis 4, PennyLane RegisterComparator 7
n = 5: correct on all 1024 inputs: True; Toffolis 5, PennyLane RegisterComparator 9
n = 6: correct on all 4096 inputs: True; Toffolis 6, PennyLane RegisterComparator 11


PennyLane's `RegisterComparator` costs $2n - 1$ in this version: it does not uncompute by measurement. Alias sampling (notebook 04) needs one comparison per PREPARE, between a $\mu$-bit random number and a $\mu$-bit table entry, so this is $\mu$ Toffolis there.

## Rotations: synthesize, or add into a phase gradient

A rotation $e^{-i\theta Z/2}$ by an arbitrary angle is not Clifford+T exactly; it is approximated by a sequence of about $3\log_2(1/\epsilon)$ T gates to operator-norm error $\epsilon$ [Ross and Selinger, [arXiv:1403.2975](https://arxiv.org/abs/1403.2975)]. When the angle comes from a register, as it does in every lookup-based algorithm, there is a cheaper way. Prepare once the phase-gradient state
$$|\phi\rangle = 2^{-b/2}\sum_{k=0}^{2^b-1} e^{-2\pi i k/2^b}|k\rangle .$$
Adding an integer $\theta$ into it permutes the basis, $|k\rangle\to|k+\theta\rangle$, yet $|\phi\rangle$ is an eigenvector of that permutation, with eigenvalue $e^{2\pi i\theta/2^b}$:

In [6]:
b = 4
for theta in (1, 3, 7):
    print(f"theta = {theta}: phase {phase_gradient_rotation(theta, b):.4f}, expected {np.exp(2j*np.pi*theta/2**b):.4f}")

theta = 1: phase 0.9239+0.3827j, expected 0.9239+0.3827j
theta = 3: phase 0.3827+0.9239j, expected 0.3827+0.9239j
theta = 7: phase -0.9239+0.3827j, expected -0.9239+0.3827j


Controlled on a qubit, that is a $Z$ rotation by $2\pi\theta/2^b$ for the price of one $b$-bit addition, $b - 2$ Toffolis with the tricks of Sanders et al. [[arXiv:2007.07391](https://arxiv.org/abs/2007.07391), App. A].

Why not synthesize? A fixed angle is one gate sequence. An angle held in a register is a superposition of $2^b$ angles, and the direct way to synthesize it is bit by bit: a rotation by $2\pi\cdot 2^j/2^b$ controlled on bit $j$, for each of the $b$ bits, each synthesized separately (a controlled rotation is in fact two rotations, so this undercounts). Compare, at angle precision $\epsilon \approx 2\pi/2^b$:

In [7]:
print(f"{'bits b':>7} {'eps':>9} {'one fixed angle, T':>19} {'register, bit by bit, T':>24} {'gradient, Toffolis':>19}")
for bits in (10, 16, 20, 30):
    eps = 2 * np.pi / 2**bits
    t = rotation_synthesis_t(eps)
    print(f"{bits:7d} {eps:9.1e} {t:19.0f} {bits * t:24.0f} {bits - 2:19d}")

 bits b       eps  one fixed angle, T  register, bit by bit, T  gradient, Toffolis
     10   6.1e-03                  22                      220                   8
     16   9.6e-05                  40                      641                  14
     20   6.0e-06                  52                     1041                  18
     30   5.9e-09                  82                     2461                  28


For a fixed angle, synthesis is the right tool. For an angle in a register, the gradient is cheaper by a factor of order $\log(1/\epsilon)$ (a Toffoli is worth 4 T gates as a temporary AND), and it composes with the rest of the circuit: a lookup writes the angle, the adder applies it, the lookup is undone. The double-factorized FeMoco walk (notebook 08) does $4N$ such rotations per step with $b = 16$ bits, $4N(b-2)$ Toffolis in all, one of the largest items of that estimate.

Newer synthesis methods do better on average: mixed fallback circuits use about $0.53\log_2(1/\epsilon) + 4.86$ T gates [Kliuchnikov et al., [arXiv:2203.10064](https://arxiv.org/abs/2203.10064), Table 1, diamond norm].

## Exercises

1. Build an $n$-bit controlled adder from `adder` by first copying $a$ into a fresh register with a controlled copy (one Toffoli per bit). What does it cost? Gidney quotes $8n + O(1)$ T gates for a controlled adder.
2. Turn `comparator` into a test of $a \le b$ by changing only the inputs. Check it on every input.
3. Subtraction: use the fact that $a - b = \overline{\bar a + b}$ to build $b \leftarrow b - a$ from `adder` and X gates. Count its Toffolis.